# Week 3 · Day 4 — Open-Source Models with Hugging Face `transformers`

This notebook is my hands-on reference for **loading and running open-source LLMs locally** with the
Hugging Face `transformers` library. It adapts the course's Colab (which assumes an NVIDIA T4 / CUDA)
to run on an **Apple-Silicon Mac (MPS)**.

## What it covers
1. **Setup & configuration** — imports, device/precision selection, Hugging Face login, and the model list.
2. **The full pipeline with one model (Llama 3.2-1B)** — tokenizer → chat template → load model → inspect architecture → generate → decode → free memory.
3. **A reusable `generate()` helper** — the same flow wrapped in a function, with streaming output.
4. **A tour of open model families** — Llama, Phi, Gemma, Qwen, DeepSeek, and how their chat formats differ.

## Key portability notes (Colab → Mac)
- **Device:** the course hard-codes `.to("cuda")`; here we detect `mps` / `cuda` / `cpu` and use a `device` variable.
- **Quantization:** `bitsandbytes` 4-bit (NF4) is **CUDA-only**, so on MPS we skip it and load in `float16`.
- **Gated models:** Llama repos require accepting Meta's license *and* a valid `HF_TOKEN`.
- **Numerics:** some models (e.g. Gemma) overflow in `float16` on MPS → load them in `float32`.

---
## 1. Setup & configuration

Imports, device/precision selection, Hugging Face authentication, and the list of model repos
we'll use. Run these cells once, top to bottom.

In [98]:
# --- Core dependencies -------------------------------------------------------
from huggingface_hub import login                 # authenticate to the HF Hub (needed for gated models)
from transformers import (
    AutoTokenizer,          # text <-> token IDs, plus chat templating
    AutoModelForCausalLM,   # decoder-only (next-token) language model
    TextStreamer,           # stream generated tokens to stdout as they arrive
    BitsAndBytesConfig,     # 4-bit / 8-bit quantization config (CUDA only)
)
import torch
import os
from dotenv import load_dotenv                     # read HF_TOKEN from a local .env file

In [99]:
# --- Pick the best available device ------------------------------------------
# The course Colab hard-codes CUDA. We detect the hardware instead so the same
# notebook runs on an Apple-Silicon Mac (MPS), an NVIDIA GPU (CUDA), or CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple-Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
else:
    device = torch.device("cpu")        # fallback

# float16 halves memory and is faster on GPU/MPS; CPU stays on float32 for stability.
dtype = torch.float16 if device.type in ("mps", "cuda") else torch.float32
print(f"Device: {device}, dtype: {dtype}")


Device: mps, dtype: torch.float16


In [100]:
# Load .env first, then log in. Llama checkpoints are gated; the Hub
# needs this token before from_pretrained can even see the model card.
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")
if not hf_token or not hf_token.startswith("hf_"):
    raise ValueError("HF_TOKEN is not set or is not a valid token")
login(hf_token, add_to_git_credential=True)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Note: Environment variable`H

In [101]:
# Llama 3.2 is a *separate* gated repo from Llama 3.1. If you 403,
# accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"  # was "=" instead of "-"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [102]:
# Chat "messages" use OpenAI-style role/content dicts. The tokenizer's chat
# template (below) converts this into each model's own prompt format.
# Roles must be lowercase: "user", "assistant"/"model", "system".
messages = [
    {"role": "user",
    "content": "Tell a joke for a room of Data Scientists"}
]


---
## 2. The full pipeline with one model (Llama 3.2-1B)

The next cells run a model **once, by hand**, so every stage of the pipeline is visible:

`quantization config → tokenizer + chat template → load model → inspect architecture → generate → decode → free memory`

> **Gated model:** Llama requires accepting Meta's license on the model card and a valid `HF_TOKEN`.
> Llama 3.2 is a *separate* gate from Llama 3.1. If you hit a 403, the tokenizer cell below
> automatically falls back to an ungated model so you can keep working.

In [103]:
# bitsandbytes 4-bit is CUDA-only. This Mac is MPS, so skip quantization
# and load in fp16. Also: the kwarg is load_in_4bit, not load_in4bit.
if device.type == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    print("Using bitsandbytes 4-bit NF4 on CUDA")
else:
    quant_config = None
    print(f"Skipping bitsandbytes (CUDA-only). Loading in {dtype} on {device}")


Skipping bitsandbytes (CUDA-only). Loading in torch.float16 on mps


In [104]:
from huggingface_hub.errors import GatedRepoError


def load_tokenizer(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    ids = tok.apply_chat_template(
        messages, tokenize=True, return_tensors="pt", add_generation_prompt=True
    ).to(device)
    return tok, ids


try:
    tokenizer, inputs = load_tokenizer(LLAMA)
    loaded_id = LLAMA
except (GatedRepoError, OSError) as exc:
    err = str(exc).lower()
    if "gated" not in err and "403" not in err:
        raise
    print(
        f"403 gated repo: {LLAMA}\n"
        "You are not on Meta's authorized list for Llama 3.2 "
        "(this is separate from Llama 3.1).\n"
        "Request access: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct\n"
        f"Continuing with ungated {DEEPSEEK}."
    )
    tokenizer, inputs = load_tokenizer(DEEPSEEK)
    loaded_id = DEEPSEEK

print(f"Tokenizer loaded: {loaded_id}")
print(f"Input tensor device: {inputs.device}, shape: {tuple(inputs.shape)}")


Tokenizer loaded: meta-llama/Llama-3.2-1B-Instruct
Input tensor device: mps:0, shape: (1, 44)


In [105]:
# The chat-templated prompt as a tensor of token IDs (on `device`).
# Values > 128000 are Llama's special/control tokens (begin_of_text, header ids, eot).
inputs

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2545,  17907,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116, 128009, 128006,  78191, 128007,    271]],
       device='mps:0')

In [106]:
# The model — Causal (next-token), not Casual.
# On MPS skip device_map="auto" and bitsandbytes; load fp16 then .to(device).
kwargs = {"token": hf_token, "torch_dtype": dtype, "low_cpu_mem_usage": True}
if quant_config is not None:
    kwargs["quantization_config"] = quant_config
    kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(loaded_id, **kwargs)
else:
    model = AutoModelForCausalLM.from_pretrained(loaded_id, **kwargs)
    model.to(device)

model.eval()
print(f"Loaded: {loaded_id}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:,.1f} MB")



Loaded: meta-llama/Llama-3.2-1B-Instruct
Memory footprint: 2,471.6 MB


In [107]:
# get_memory_footprint() reports the model's RAM/VRAM use (bytes -> MB).
# Handy for comparing fp16 vs quantized sizes across models.
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 2,471.6 MB


### Inspecting the architecture

Printing the model object shows the **decoder-only Transformer** stack:

`embed_tokens` (token IDs → vectors) → **N × decoder layers** (self-attention + MLP) → final norm → `lm_head` (hidden state → vocabulary logits).

For Llama 3.2-1B you'll see 16 `LlamaDecoderLayer`s and a 2048-dim hidden size.

In [108]:
# Print the module tree: embeddings, the repeated decoder layers, and the LM head.
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [109]:
# generate() autoregressively produces up to max_new_tokens new tokens.
# `inputs` already contains the chat-templated prompt token IDs.
# The attention_mask / pad_token warning is expected here because we pass raw inputs
# (the reusable generate() helper later sets both explicitly).
outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]   # 1-D tensor: the prompt tokens followed by the newly generated ones

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   2545,  17907,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
           315,   2956,  57116, 128009, 128006,  78191, 128007,    271,     32,
          3130,   2539,    315,    828,  14248,     11,    358,   3077,   2751,
           832,    430,    596,    704,    315,    420,   1917,     13,   5810,
           433,   5900,   1473,  10445,   1550,    279,  31649,   1646,    733,
           311,  15419,   1980,  18433,    433,    574,  20558,    311,   7168,
          1202,  21958,   2268,   9149,   3987,    430,    832,  18809,    279,
          4121,    323,  12716,   1063,  43214,    311,    279,   3130,  16715,
        128009], device='mps:0')

In [110]:
# Convert token IDs back into text. Without skip_special_tokens you can see the
# chat control tokens (<|begin_of_text|>, <|eot_id|>, ...) the template inserted.
tokenizer.decode(outputs[0])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 09 Sep 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nA room full of data scientists, I've got one that's out of this world. Here it goes:\n\nWhy did the regression model go to therapy?\n\nBecause it was struggling to predict its emotions!\n\n(I hope that one fits the bill and brings some laughter to the room!)<|eot_id|>"

In [111]:
import gc

# Safe to re-run: del only names that still exist.
for name in ("model", "inputs", "tokenizer", "outputs"):
    if name in globals():
        del globals()[name]

gc.collect()
# torch.device has no empty_cache(). CUDA vs MPS:
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()


---
## 3. A reusable `generate()` helper

Everything from section 2, wrapped into one function so we can try many models quickly.
It adds two conveniences over the manual flow:
- **`TextStreamer`** — prints tokens as they're generated instead of waiting for the full response.
- **explicit `attention_mask` + `pad_token_id`** — silences the warnings above and gives reliable results.

The `model_dtype` argument lets us override the global `dtype` per model (needed for Gemma on MPS).

In [112]:
def generate(model, messages, quant=True, max_new_tokens=80, model_dtype=None):
    tokenizer = AutoTokenizer.from_pretrained(model, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(device)
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)
    streamer = TextStreamer(tokenizer)

    # Causal (next-token), not Casual. Skip 4-bit unless we actually built a CUDA config.
    # model_dtype override: fp16 overflows to inf/nan for some models (e.g. Gemma) on MPS,
    # which crashes torch.multinomial. Pass float32 for those.
    load_dtype = model_dtype or dtype
    use_quant = quant and quant_config is not None
    kwargs = {"token": hf_token, "torch_dtype": load_dtype, "low_cpu_mem_usage": True}
    if use_quant:
        kwargs["quantization_config"] = quant_config
        kwargs["device_map"] = "auto"
        mdl = AutoModelForCausalLM.from_pretrained(model, **kwargs)
    else:
        mdl = AutoModelForCausalLM.from_pretrained(model, **kwargs)
        mdl.to(device)

    outputs = mdl.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# apply_chat_template needs at least one turn; an empty list raises IndexError.
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
]


---
## 4. A tour of open model families

The same `generate()` call now runs across several model families. Watch how each model's
**chat template and special tokens differ** in the streamed output:

| Model | Repo | Format / notes |
|-------|------|----------------|
| Phi-4-mini | `microsoft/Phi-4-mini-instruct` | `<\|user\|> … <\|assistant\|>` tags |
| Gemma 3 (270m) | `google/gemma-3-270m-it` | lowercase roles; needs **fp32** on MPS |
| Qwen3-4B | `Qwen/Qwen3-4B-Instruct-2507` | ChatML: `<\|im_start\|>` / `<\|im_end\|>` |
| DeepSeek-R1-Distill | `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | reasoning model; emits `<think> … </think>` |

In [113]:
generate(PHI, messages)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<|user|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a data science-themed joke for you:

Why don't data scientists like to play hide and seek?

Because good luck hiding when you have a dataset with 100% accuracy!<|end|>


"Tell a joke for a room of Data ScientistsSure, here's a data science-themed joke for you:\n\nWhy don't data scientists like to play hide and seek?\n\nBecause good luck hiding when you have a dataset with 100% accuracy!"

### Gemma (Google)

Gemma also requires accepting its license on the Hub. Two gotchas this notebook handles:
- Its chat template **only accepts lowercase roles** (`user` / `model`) — `"User"` raises a `TemplateError`.
- It **overflows in float16 on MPS** (→ `inf`/`nan` → `torch.multinomial` crash), so we load it in `float32`
  via `model_dtype=torch.float32`.

In [114]:
# Role must be lowercase "user"; Gemma's chat template rejects "User" with a TemplateError.
messages = [
    {"role": "user",
    "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

# Gemma overflows in fp16 on MPS (inf/nan -> multinomial crash). It's tiny, so load fp32.
generate(GEMMA, 
        messages,
        quant=False,
        model_dtype=torch.float32)


<bos><start_of_turn>user
Tell a light-hearted joke for a room of Data Scientists<end_of_turn>
<start_of_turn>model
Why don't data scientists ever get a good cup of coffee? 
<end_of_turn>


"user\nTell a light-hearted joke for a room of Data Scientists\nmodel\nWhy don't data scientists ever get a good cup of coffee? \n"

In [115]:
generate(QWEN, messages)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<|im_start|>user
Tell a light-hearted joke for a room of Data Scientists<|im_end|>
<|im_start|>assistant
Sure! Here's a light-hearted, data-savvy joke just for the Data Scientists in the room:

---

Why did the data scientist break up with the regression model?

Because it was always *correlated* with the same old pattern — and never really *linear* in its feelings!

😄

*(Bonus points if you chuckle and then quietly check if your model has a high R-s


"user\nTell a light-hearted joke for a room of Data Scientists\nassistant\nSure! Here's a light-hearted, data-savvy joke just for the Data Scientists in the room:\n\n---\n\nWhy did the data scientist break up with the regression model?\n\nBecause it was always *correlated* with the same old pattern — and never really *linear* in its feelings!\n\n😄\n\n*(Bonus points if you chuckle and then quietly check if your model has a high R-s"

In [116]:
generate(DEEPSEEK, 
         messages,
         quant=False,
         max_new_tokens = 500)

<｜begin▁of▁sentence｜><｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜><think>
Okay, so I need to come up with a light-hearted joke for a room of data scientists. Hmm, data scientists are often seen as technical and serious, so I need to make the joke relatable and funny without being too technical. Let me think about the common phrases or situations data scientists face.

One thing that comes to mind is the data generation part. Data scientists collect a lot of data, so maybe something about data being collected or processed. Maybe something about the data being used for something, like analytics or something else. Also, sometimes data scientists talk about their work being a puzzle or a challenge.

Wait, there's a classic joke about data being the king of the database, right? It's something like, "Data is the king of the database, and the king of the world." That's a classic, but maybe I can tweak it to make it more relatable for a room full of data scientis

'<｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜><think>\nOkay, so I need to come up with a light-hearted joke for a room of data scientists. Hmm, data scientists are often seen as technical and serious, so I need to make the joke relatable and funny without being too technical. Let me think about the common phrases or situations data scientists face.\n\nOne thing that comes to mind is the data generation part. Data scientists collect a lot of data, so maybe something about data being collected or processed. Maybe something about the data being used for something, like analytics or something else. Also, sometimes data scientists talk about their work being a puzzle or a challenge.\n\nWait, there\'s a classic joke about data being the king of the database, right? It\'s something like, "Data is the king of the database, and the king of the world." That\'s a classic, but maybe I can tweak it to make it more relatable for a room full of data scientists. Perhaps 

---
## Conclusion & quick reference

**The core local-inference recipe:**
1. Pick `device` / `dtype` (MPS · CUDA · CPU).
2. `AutoTokenizer.from_pretrained(...)` → `apply_chat_template(messages, add_generation_prompt=True)`.
3. `AutoModelForCausalLM.from_pretrained(..., torch_dtype=dtype)`, then `.to(device)` (no quantization on MPS).
4. `model.generate(input_ids, attention_mask=..., max_new_tokens=...)`.
5. `tokenizer.decode(outputs[0])`.
6. Free memory: `del` the objects + `gc.collect()` + `torch.mps/cuda.empty_cache()`.

**Gotchas this notebook solves (Colab → Mac):**
- `AutoModelForCausalLM`, **not** `Casual` — easy spelling trap.
- `bitsandbytes` 4-bit (NF4) is **CUDA-only** → skip on MPS and load in `float16`.
- Llama is **gated** → accept the license + set `HF_TOKEN`; fall back to an ungated model on a 403.
- Chat roles must be **lowercase** → Gemma raises `TemplateError` on `"User"`.
- Gemma **overflows in fp16 on MPS** → load in `float32` to avoid `inf`/`nan` in `torch.multinomial`.
- Always pass `attention_mask` and `pad_token_id` to `generate()` for reliable results.

**Ideas for next time:** vary the prompt and `max_new_tokens`, compare memory footprints across
models, add `do_sample`/`temperature`/`top_p` for creativity, and — on a CUDA box — re-enable
4-bit quantization to fit larger models.